In [1]:
from pyspark.sql import SparkSession
from pyspark.sql.functions import *

In [2]:
spark = SparkSession.builder.master("local[*]")\
                    .appName("broadcastHashJoin")\
                    .getOrCreate()
spark

### Customer data and Temp view

In [3]:
customer_data = [
    (1, 'manish', 'patna', '30-05-2022'),
    (2, 'vikash', 'kolkata', '12-03-2023'),
    (3, 'nikita', 'delhi', '25-06-2023'),
    (4, 'rahul', 'ranchi', '24-03-2023'),
    (5, 'mahesh', 'jaipur', '22-03-2023'),
    (6, 'prantosh', 'kolkata', '18-10-2022'),
    (7, 'raman', 'patna', '30-12-2022'),
    (8, 'prakash', 'ranchi', '24-02-2023'),
    (9, 'ragini', 'kolkata', '03-03-2023'),
    (10, 'raushan', 'jaipur', '05-02-2023')
]
customer_schema = ['customer_id', 'custmer_name', 'address', 'date_of_joining']
customer_df = spark.createDataFrame(data=customer_data, schema=customer_schema)
customer_df.createOrReplaceTempView("customer_tbl")


### sales data and temp view

In [4]:
sales_data = [
    (1, 22, 10, "01-06-2022"),
    (1, 27, 5, "03-02-2023"),
    (2, 5, 3, "01-08-2023"),
    (5, 22, 1, "22-03-2023"),
    (7, 22, 4, "03-02-2023"),
    (9, 5, 6, "03-03-2023"),
    (2, 1, 12, "15-06-2023"),
    (1, 56, 2, "25-06-2023"),
    (5, 12, 5, "15-04-2023"),
    (11, 12, 76, "12-03-2023")
]
sales_schema = ['customer_id','product_id', 'quantity', 'date_of_purchase']
sales_df = spark.createDataFrame(data=sales_data, schema=sales_schema)
sales_df.createOrReplaceTempView("sales_tbl")


### join

In [5]:
sort_merge_df = customer_df.join(sales_df, customer_df["customer_id"]==sales_df["customer_id"], "inner")

In [6]:
sort_merge_df.show()

+-----------+------------+-------+---------------+-----------+----------+--------+----------------+
|customer_id|custmer_name|address|date_of_joining|customer_id|product_id|quantity|date_of_purchase|
+-----------+------------+-------+---------------+-----------+----------+--------+----------------+
|          1|      manish|  patna|     30-05-2022|          1|        22|      10|      01-06-2022|
|          1|      manish|  patna|     30-05-2022|          1|        27|       5|      03-02-2023|
|          1|      manish|  patna|     30-05-2022|          1|        56|       2|      25-06-2023|
|          2|      vikash|kolkata|     12-03-2023|          2|         5|       3|      01-08-2023|
|          2|      vikash|kolkata|     12-03-2023|          2|         1|      12|      15-06-2023|
|          5|      mahesh| jaipur|     22-03-2023|          5|        22|       1|      22-03-2023|
|          5|      mahesh| jaipur|     22-03-2023|          5|        12|       5|      15-04-2023|


In [7]:
sort_merge_df.explain()

== Physical Plan ==
AdaptiveSparkPlan isFinalPlan=false
+- SortMergeJoin [customer_id#0L], [customer_id#4L], Inner
   :- Sort [customer_id#0L ASC NULLS FIRST], false, 0
   :  +- Exchange hashpartitioning(customer_id#0L, 200), ENSURE_REQUIREMENTS, [plan_id=145]
   :     +- Filter isnotnull(customer_id#0L)
   :        +- Scan ExistingRDD[customer_id#0L,custmer_name#1,address#2,date_of_joining#3]
   +- Sort [customer_id#4L ASC NULLS FIRST], false, 0
      +- Exchange hashpartitioning(customer_id#4L, 200), ENSURE_REQUIREMENTS, [plan_id=146]
         +- Filter isnotnull(customer_id#4L)
            +- Scan ExistingRDD[customer_id#4L,product_id#5L,quantity#6L,date_of_purchase#7]




## Join in Spark
**Potential interview question**
1. How join works ?
2. Why do we need join ?
3. What to do after joining two tables ?
4. What if two tables have same column name ?
5. Join on two or more columns ?
6. Types of join ?

### DATA SET

In [8]:
customer_data = [(1,'manish','patna',"30-05-2022"),
(2,'vikash','kolkata',"12-03-2023"),
(3,'nikita','delhi',"25-06-2023"),
(4,'rahul','ranchi',"24-03-2023"),
(5,'mahesh','jaipur',"22-03-2023"),
(6,'prantosh','kolkata',"18-10-2022"),
(7,'raman','patna',"30-12-2022"),
(8,'prakash','ranchi',"24-02-2023"),
(9,'ragini','kolkata',"03-03-2023"),
(10,'raushan','jaipur',"05-02-2023")]

customer_schema=['customer_id','customer_name','address','date_of_joining']


sales_data = [(1,22,10,"01-06-2022"),
(1,27,5,"03-02-2023"),
(2,5,3,"01-06-2023"),
(5,22,1,"22-03-2023"),
(7,22,4,"03-02-2023"),
(9,5,6,"03-03-2023"),
(2,1,12,"15-06-2023"),
(1,56,2,"25-06-2023"),
(5,12,5,"15-04-2023"),
(11,12,76,"12-03-2023")]

sales_schema=['customer_id','product_id','quantity','date_of_purchase']


product_data = [(1, 'fanta',20),
(2, 'dew',22),
(5, 'sprite',40),
(7, 'redbull',100),
(12,'mazza',45),
(22,'coke',27),
(25,'limca',21),
(27,'pepsi',14),
(56,'sting',10)]

product_schema=['id','name','price']

In [9]:
customer_df.show()

+-----------+------------+-------+---------------+
|customer_id|custmer_name|address|date_of_joining|
+-----------+------------+-------+---------------+
|          1|      manish|  patna|     30-05-2022|
|          2|      vikash|kolkata|     12-03-2023|
|          3|      nikita|  delhi|     25-06-2023|
|          4|       rahul| ranchi|     24-03-2023|
|          5|      mahesh| jaipur|     22-03-2023|
|          6|    prantosh|kolkata|     18-10-2022|
|          7|       raman|  patna|     30-12-2022|
|          8|     prakash| ranchi|     24-02-2023|
|          9|      ragini|kolkata|     03-03-2023|
|         10|     raushan| jaipur|     05-02-2023|
+-----------+------------+-------+---------------+



In [10]:
sales_df.show()

+-----------+----------+--------+----------------+
|customer_id|product_id|quantity|date_of_purchase|
+-----------+----------+--------+----------------+
|          1|        22|      10|      01-06-2022|
|          1|        27|       5|      03-02-2023|
|          2|         5|       3|      01-08-2023|
|          5|        22|       1|      22-03-2023|
|          7|        22|       4|      03-02-2023|
|          9|         5|       6|      03-03-2023|
|          2|         1|      12|      15-06-2023|
|          1|        56|       2|      25-06-2023|
|          5|        12|       5|      15-04-2023|
|         11|        12|      76|      12-03-2023|
+-----------+----------+--------+----------------+



In [12]:
customer_df.join(sales_df, customer_df["customer_id"]==sales_df["customer_id"], "inner").show()

+-----------+------------+-------+---------------+-----------+----------+--------+----------------+
|customer_id|custmer_name|address|date_of_joining|customer_id|product_id|quantity|date_of_purchase|
+-----------+------------+-------+---------------+-----------+----------+--------+----------------+
|          1|      manish|  patna|     30-05-2022|          1|        22|      10|      01-06-2022|
|          1|      manish|  patna|     30-05-2022|          1|        27|       5|      03-02-2023|
|          1|      manish|  patna|     30-05-2022|          1|        56|       2|      25-06-2023|
|          2|      vikash|kolkata|     12-03-2023|          2|         5|       3|      01-08-2023|
|          2|      vikash|kolkata|     12-03-2023|          2|         1|      12|      15-06-2023|
|          5|      mahesh| jaipur|     22-03-2023|          5|        22|       1|      22-03-2023|
|          5|      mahesh| jaipur|     22-03-2023|          5|        12|       5|      15-04-2023|


In [ ]:
customer_df.join(sales_df, customer_df["customer_id"]==sales_df["customer_id"], "inner").select(sales_df["product_id"]).show()

+----------+
|product_id|
+----------+
|        22|
|        27|
|        56|
|         5|
|         1|
|        22|
|        12|
|        22|
|         5|
+----------+

